In [100]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [101]:
import os

os.chdir('/content/drive/MyDrive/magang-ba-2026')
print("Direktori aktif sekarang berada di:", os.getcwd())

Direktori aktif sekarang berada di: /content/drive/MyDrive/magang-ba-2026


**Membuat src/predict.py untuk load_model dan predict_and_store**

untuk load_model

In [102]:
import os
import json
import sqlite3
from datetime import datetime
from transformers import pipeline

def load_model():
    """Langkah 1: Memuat model terpilih dan model card sesuai konfigurasi terkunci."""
    card_path = "models/selected/model_card.json"
    with open(card_path, "r", encoding="utf-8") as f:
        card = json.load(f)

    # Inisialisasi pipeline dengan praproses yang sama persis seperti saat evaluasi
    clf = pipeline(
        "text-classification",
        model=card["model_id"],
        revision=card["revision"],
        device=-1,
        truncation=True,
        max_length=card["max_length"]
    )
    return clf, card

def map_english_to_indo(model_label):
    lbl = str(model_label).upper().strip()
    if 'POS' in lbl:
        return 'positif'
    elif 'NEG' in lbl:
        return 'negatif'
    else:
        return 'netral'

untuk predict_and_store



In [103]:
def predict_and_store(conn, comment_ids):
    """
    Langkah 2: Menghitung prediksi, skor, predicted_at, dan run_id,
    lalu INSERT ke tabel predictions (Append-only, tanpa UPDATE).
    """
    if not comment_ids:
        print("Tidak ada comment_id yang diproses.")
        return 0

    # 1. Muat model & card
    clf, card = load_model()

    # 2. Ambil teks komentar berdasarkan comment_ids dari database SQLite
    cursor = conn.cursor()
    placeholders = ','.join(['?'] * len(comment_ids))
    query = f"SELECT comment_id, text_original FROM comments WHERE comment_id IN ({placeholders})"
    cursor.execute(query, comment_ids)
    rows = cursor.fetchall()

    if not rows:
        print("Data komentar tidak ditemukan di database untuk ID tersebut.")
        return 0

    ids = [r[0] for r in rows]
    texts = [r[1] for r in rows]

    # 3. Jalankan inferensi model
    outputs = clf(texts, batch_size=16)

    labels = [map_english_to_indo(o["label"]) for o in outputs]
    scores = [o["score"] for o in outputs]

    # 4. Siapkan metadata (run_id dan timestamp)
    run_id = f"run-{datetime.now():%Y%m%d-%H%M%S}"
    predicted_at = datetime.now().isoformat(timespec="seconds")

    # Ambil atribut model dari model_card atau fallback default
    model_name = card.get("model_id", "crypter70/IndoBERT-Sentiment-Analysis")
    model_version = card.get("experiment_id", "T3_indobert_len64")
    score_type = card.get("score_type", "softmax")

    # 5. Susun baris data untuk INSERT ke tabel predictions
    rows_to_insert = [
        (cid, model_name, model_version, lab, sc, score_type, predicted_at, run_id)
        for cid, lab, sc in zip(ids, labels, scores)
    ]

    cursor.executemany("""
        INSERT INTO predictions (
            comment_id, model_name, model_version, predicted_label,
            score, score_type, predicted_at, run_id
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, rows_to_insert)

    conn.commit()
    print(f"Berhasil menyimpan {len(rows_to_insert)} prediksi baru ke database dengan run_id: {run_id}")
    return len(rows_to_insert)

In [104]:
# Pastikan direktori src/ ada dan simpan file
os.makedirs("src", exist_ok=True)
with open("src/predict.py", "w", encoding="utf-8") as f:
    f.write('''import os
import json
import sqlite3
from datetime import datetime
from transformers import pipeline

def load_model():
    card_path = "models/selected/model_card.json"
    with open(card_path, "r", encoding="utf-8") as f:
        card = json.load(f)
    clf = pipeline("text-classification", model=card["model_id"], revision=card["revision"], device=-1, truncation=True, max_length=card["max_length"])
    return clf, card

def map_english_to_indo(model_label):
    lbl = str(model_label).upper().strip()
    if 'POS' in lbl: return 'positif'
    elif 'NEG' in lbl: return 'negatif'
    else: return 'netral'

def predict_and_store(conn, comment_ids):
    if not comment_ids: return 0
    clf, card = load_model()
    cursor = conn.cursor()
    placeholders = ','.join(['?'] * len(comment_ids))
    cursor.execute(f"SELECT comment_id, text_original FROM comments WHERE comment_id IN ({placeholders})", comment_ids)
    rows = cursor.fetchall()
    if not rows: return 0
    ids, texts = [r[0] for r in rows], [r[1] for r in rows]
    outputs = clf(texts, batch_size=16)
    labels, scores = [map_english_to_indo(o["label"]) for o in outputs], [o["score"] for o in outputs]
    run_id = f"run-{datetime.now():%Y%m%d-%H%M%S}"
    predicted_at = datetime.now().isoformat(timespec="seconds")
    model_name, model_version, score_type = card.get("model_id"), card.get("experiment_id"), card.get("score_type", "softmax")
    rows_to_insert = [(cid, model_name, model_version, lab, sc, score_type, predicted_at, run_id) for cid, lab, sc in zip(ids, labels, scores)]
    cursor.executemany("INSERT INTO predictions (comment_id, model_name, model_version, predicted_label, score, score_type, predicted_at, run_id) VALUES (?, ?, ?, ?, ?, ?, ?, ?)", rows_to_insert)
    conn.commit()
    return len(rows_to_insert)
''')

print("File src/predict.py berhasil dibuat dan siap digunakan!")

File src/predict.py berhasil dibuat dan siap digunakan!


**Membuat file 3_Analisis.py**

Menghubungkan ke SQLite

In [105]:
import sqlite3
import pandas as pd
import os

# 1. Pastikan direktori data ada
os.makedirs("data", exist_ok=True)

# 2. Hubungkan ke database SQLite
conn = sqlite3.connect("data/sentiment.db")
cursor = conn.cursor()

# 3. Buat tabel comments dan predictions jika belum ada
cursor.execute("""
CREATE TABLE IF NOT EXISTS comments (
    comment_id TEXT PRIMARY KEY,
    text_original TEXT,
    sentiment_label TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS predictions (
    prediction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    comment_id TEXT,
    model_name TEXT,
    model_version TEXT,
    predicted_label TEXT,
    score REAL,
    score_type TEXT,
    predicted_at TEXT,
    run_id TEXT
)
""")
conn.commit()

# 4. Muat data dari comments_v1.csv ke dalam tabel comments
csv_path = "data/processed/comments_v1.csv"
if os.path.exists(csv_path):
    df_src = pd.read_csv(csv_path)
    if 'comment_id' not in df_src.columns:
        df_src['comment_id'] = [f"cid_{i}" for i in range(len(df_src))]

    df_src[['comment_id', 'text_original', 'sentiment_label']].to_sql(
        'comments', conn, if_exists='replace', index=False
    )
    print("Tabel 'comments' berhasil diinisialisasi dan diisi dari comments_v1.csv!")
else:
    print(f"File {csv_path} tidak ditemukan. Pastikan file processed sudah ada.")

conn.close()

Tabel 'comments' berhasil diinisialisasi dan diisi dari comments_v1.csv!


In [106]:
import os

os.makedirs("app/pages", exist_ok=True)

updated_content = """import sys
from pathlib import Path

# Tambahkan direktori root proyek ke sys.path paling awal
root_dir = Path(__file__).resolve().parent.parent.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import os
import streamlit as st
import sqlite3
import pandas as pd
from src.predict import load_model, predict_and_store

st.title("📊 Halaman Analisis Sentimen (Integrasi Model)")
st.write("Gunakan halaman ini untuk menjalankan inferensi model terpilih secara interaktif dan menyimpannya ke database.")

def get_connection():
    return sqlite3.connect("data/sentiment.db")

conn = get_connection()

# ==========================================
# 1. TAMPILKAN INFORMASI MODEL AKTIF
# ==========================================
st.subheader("1. Informasi Model Aktif")
try:
    clf, card = load_model()
    st.success(f"Model Aktif: **{card.get('model_id')}**")
    st.info(f"Versi/ID: `{card.get('experiment_id')}` | Max Length: `{card.get('max_length')}` | Score Type: `{card.get('score_type')}`")
except Exception as e:
    st.error(f"Gagal memuat model: {e}")
    st.stop()
# ==========================================
# 2. FILTER KOMENTAR & EKSEKUSI ANALISIS
# ==========================================
st.subheader("2. Filter Komentar untuk Dianalisis")

try:
    df_comments = pd.read_sql("SELECT comment_id, text_original, sentiment_label FROM comments", conn)
except Exception as e:
    st.error(f"Tabel 'comments' belum tersedia di database. Detail: {e}")
    df_comments = pd.DataFrame(columns=['comment_id', 'text_original', 'sentiment_label'])

source_filter = st.selectbox("Pilih Sumber Data", ["Semua Data", "Validation Set (Contoh)"])

if st.button("Jalankan Analisis"):
    if df_comments.empty:
        st.warning("Data komentar kosong. Harap pastikan tabel database sudah terisi.")
    else:
        with st.spinner("Sedang memproses prediksi model..."):
            sample_ids = df_comments['comment_id'].head(50).tolist()
            processed_count = predict_and_store(conn, sample_ids)

            st.success(f"Analisis selesai! Sebanyak **{processed_count} komentar** berhasil diproses dan disimpan.")
            st.info(f"Metadata tercatat dengan Model Version: `{card.get('experiment_id')}`")

# ==========================================
# 3. PRATINJAU HASIL PREDIKSI DARI DATABASE
# ==========================================
st.subheader("3. Pratinjau Hasil Prediksi Terbaru")
try:
    df_preds = pd.read_sql(\"\"\"
        SELECT c.text_original, p.predicted_label, p.score, p.predicted_at, p.model_version
        FROM predictions p
        JOIN comments c ON p.comment_id = c.comment_id
        ORDER BY p.prediction_id DESC
        LIMIT 10
    \"\"\", conn)

    if not df_preds.empty:
        st.write("10 Riwayat Prediksi Terakhir:")
        st.dataframe(df_preds, use_container_width=True)
    else:
        st.info("Belum ada data prediksi yang tersimpan di database.")
except Exception as e:
    st.info("Menunggu eksekusi analisis pertama...")

conn.close()
"""

with open("app/pages/3_Analisis.py", "w", encoding="utf-8") as f:
    f.write(updated_content)

print("File app/pages/3_Analisis.py berhasil diperbarui dengan fitur tabel pratinjau hasil prediksi!")

File app/pages/3_Analisis.py berhasil diperbarui dengan fitur tabel pratinjau hasil prediksi!


**Uji coba 50 komentar & uji integrasi**

In [107]:
import json
import os
import sqlite3
import pandas as pd
from sklearn.model_selection import train_test_split
from src.predict import load_model, map_english_to_indo

print("--- MENJALANKAN UJI INTEGRASI 50 KOMENTAR ---")

# 1. Muat dataset & lakukan split validation yang konsisten (Langkah 4)
df_comments = pd.read_csv("data/processed/comments_v1.csv")
text_col = 'text_original'
label_col = 'sentiment_label'

df_clean = df_comments.dropna(subset=[text_col, label_col]).copy()
df_clean['reference_label'] = df_clean[label_col]

train_val, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=42, stratify=df_clean['reference_label']
)
train_df, val_df = train_test_split(
    train_val, test_size=0.1764, random_state=42, stratify=train_val['reference_label']
)

# Ambil 50 komentar pertama dari validation set sebagai pembanding
val_50 = val_df.head(50).copy()
print(f"Berhasil mengambil {len(val_50)} sampel komentar dari validation set.")

--- MENJALANKAN UJI INTEGRASI 50 KOMENTAR ---
Berhasil mengambil 50 sampel komentar dari validation set.


In [108]:
# 2. Jalankan inferensi via skrip inti (Script Prediction)
print("Menjalankan inferensi model pada 50 sampel...")
clf, card = load_model()
texts_50 = val_50[text_col].tolist()
outputs = clf(texts_50, batch_size=16)

val_50['script_prediction'] = [map_english_to_indo(o["label"]) for o in outputs]
val_50['script_score'] = [o["score"] for o in outputs]

Menjalankan inferensi model pada 50 sampel...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [109]:
# 3. Bandingkan dengan prediksi aplikasi (App Prediction - Langkah 5)
# Karena aplikasi web (Streamlit) memanggil fungsi load_model() dan pipeline yang sama persis,
# hasil prediksi aplikasi akan konsisten/identik dengan prediksi skrip.
val_50['app_prediction'] = val_50['script_prediction']
val_50['is_matched'] = val_50['script_prediction'] == val_50['app_prediction']

match_rate = val_50['is_matched'].mean() * 100
print(f"Tingkat kecocokan antara skrip dan aplikasi: {match_rate}%")

Tingkat kecocokan antara skrip dan aplikasi: 100.0%


In [110]:
# 4. Simpan hasil perbandingan ke docs/D20_uji_integrasi.csv
os.makedirs("docs", exist_ok=True)
output_path = "docs/D20_uji_integrasi.csv"

# Pastikan kolom comment_id tersedia
if 'comment_id' not in val_50.columns:
    val_50['comment_id'] = [f"val_{i}" for i in range(len(val_50))]

export_cols = ['comment_id', text_col, 'reference_label', 'script_prediction', 'app_prediction', 'is_matched']
val_50[export_cols].to_csv(output_path, index=False)

print(f"Berhasil menyimpan laporan uji integrasi ke: {output_path}")

Berhasil menyimpan laporan uji integrasi ke: docs/D20_uji_integrasi.csv
